# Kaggle GPU smoke test
This notebook checks each CUDA device independently. Two T4 GPUs are not combined into one 30 GB device.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / 'src').is_dir():
    raise RuntimeError('Run this notebook with the GaussTwin repository as the working directory.')
sys.path.insert(0, str(repo_root / 'src'))

from gausstwin.utils.environment import collect_environment, format_environment

report = collect_environment()
print(format_environment(report))
if not report['cuda_available']:
    raise RuntimeError('CUDA is unavailable. Enable a Kaggle GPU accelerator and restart the kernel.')

In [ ]:
import torch

for index in range(report['gpu_count']):
    device = torch.device(f'cuda:{index}')
    # Four float32 values: a deliberately tiny per-device operation.
    value = (torch.arange(4, dtype=torch.float32, device=device) + 1).sum()
    torch.cuda.synchronize(index)
    print(f'GPU {index} passed: sum={value.item():.1f} on {device}')

if report['gpu_count'] < 2:
    print('Warning: this session does not expose both expected T4 devices.')
print('Each GPU was tested independently; no cross-device memory pooling was used.')